# 04 — The per-night table (handoff §1b)

    weather_hours         = |W ∩ N|            W = WEATHER bit (from 2026-02-25) or EFD criteria (before)
    on_sky_hours          = |S ∩ N|            S = visit coverage, gaps > gap_min are loss
    nonweather_loss_hours = |N| − |W ∪ S|      fault + idle, reported as "fault" for the talks (Bruno)

Then the cloud step decided by Bruno on 2026-10-18 (option 2): before the WEATHER bit, `closed_unexplained_hours` (dome shut
or telescope at horizon with covers on, no EFD criterion met) moves into `weather_hours` **only** on nights where the night
report (Source 2) gives P(weather) ≥ 0.5. Those nights carry `weather_source = "EFD criteria + night report (cloud)"` and the
moved hours in `cloud_reassigned_hours`. The `label` is the handoff's presentation rule, not a definition.

Output: `data/processed/per_night.csv` + `.parquet`; `per_night_v0.csv` is the table before the cloud step.

In [ ]:
import sys, logging
from pathlib import Path
import numpy as np, pandas as pd
sys.path.insert(0, "..")
from time_loss import DAY_OBS_MIN, DAY_OBS_MAX, GAP_MIN_MINUTES, GAP_MIN_SENSITIVITY, DAY_OBS_WEATHER_BIT_START
pd.set_option("display.width", 220); pd.set_option("display.max_rows", 300); pd.set_option("display.max_colwidth", 80)
logging.basicConfig(level=logging.WARNING)
for name in ("httpx", "numexpr", "time_loss.access"):
    logging.getLogger(name).setLevel(logging.WARNING)
STEP0 = Path("../data/step0"); CACHE = Path("../data/cache"); PROC = Path("../data/processed")
for p in (STEP0, CACHE, PROC): p.mkdir(parents=True, exist_ok=True)

In [ ]:
# ROSP-24 ordered accounting (2026-10-26): one per-minute classifier over the −12° night.
# The arithmetic lives in time_loss/classify.py and is driven by python/run_rosp24_accounting.py;
# this notebook only runs it and shows the result. Inputs are the cached step-0 tables.
import runpy
runpy.run_path('../python/run_rosp24_accounting.py', run_name='__main__')


In [ ]:
t = pd.read_csv(PROC / 'per_night.csv')
cols = ['night_hours', 'weather_hours', 'engineering_hours', 'focus_alignment_hours', 'nonweather_loss_hours', 'dynalene_hours', 'on_sky_hours', 'startup_hours', 'unattributed_hours', 'weather_masked_fault_hours', 'idle_bit_hours', 'report_cloud_hours']
print(t[cols].sum().round(1).to_dict()); print(t.label.value_counts().to_dict())
t.groupby('weather_source')[['night_hours', 'weather_hours', 'on_sky_hours', 'engineering_hours', 'nonweather_loss_hours']].agg(['sum', 'count']).round(1)


## Monthly summary

In [ ]:
m = pd.read_csv(PROC / 'per_month.csv', index_col=0); m


## The full table, for spot checks against the human tables

In [ ]:
cols = ['day_obs', 'night_hours', 'weather_hours', 'weather_source', 'engineering_hours', 'on_sky_hours', 'nonweather_loss_hours', 'dynalene_hours', 'startup_hours', 'unattributed_hours', 'label', 'weather_windows_utc']
t[cols]
